# MNIST digit classifier

Small fully connected net that learns to read handwritten digits.

In [ ]:
# imports
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# use gpu if we got one
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device:", device)

using device: cpu


In [ ]:
# grab mnist, 28x28 grayscale digit images

# ToTensor converts to tensor + scales pixels to 0-1
transform = transforms.ToTensor()

# 60k training images, downloads once then caches
train_dataset = torchvision.datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

# 10k test images, only for checking, never trained on
test_dataset = torchvision.datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# dataloader batches + shuffles for us
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("train images:", len(train_dataset))
print("test images:", len(test_dataset))

train images: 60000
test images: 10000


In [ ]:
# the network, just a stack of fully connected layers
class DigitClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        # 28x28 image flattened into one long vector = 784
        input_size = 28 * 28

        # 784 pixels in, 128 features out
        self.hidden1 = nn.Linear(input_size, 128)

        # 128 down to 64
        self.hidden2 = nn.Linear(128, 64)

        # 64 down to 10, one score per digit 0-9
        self.output = nn.Linear(64, 10)

        # relu = the nonlinearity, lets the net learn more than straight lines
        self.relu = nn.ReLU()

    def forward(self, x):
        # flatten each image, batch stays the same size
        x = x.view(x.size(0), -1)
        x = self.relu(self.hidden1(x))
        x = self.relu(self.hidden2(x))
        # no activation on the last layer, loss function takes care of that
        x = self.output(x)
        return x

model = DigitClassifier().to(device)
print(model)

DigitClassifier(
  (hidden1): Linear(in_features=784, out_features=128, bias=True)
  (hidden2): Linear(in_features=128, out_features=64, bias=True)
  (output): Linear(in_features=64, out_features=10, bias=True)
  (relu): ReLU()
)


In [ ]:

# cross entropy = standard loss for classification
# heads up, this already does softmax internally, thats why output layer has no activation
criterion = nn.CrossEntropyLoss()

# adam does the actual gradient descent step
# lr = how big of a step it takes each update
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:

# 1 epoch = one full pass over all training data
num_epochs = 5

for epoch in range(num_epochs):
    model.train()

    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        # reset gradients, pytorch adds to them by default otherwise
        optimizer.zero_grad()

        # run images through the net
        outputs = model(images)

        # how wrong were we
        loss = criterion(outputs, labels)

        # backprop, pytorch works out all the gradients for us
        loss.backward()

        # nudge the weights to lower the loss
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"epoch {epoch + 1}/{num_epochs}, avg loss: {avg_loss:.4f}")

epoch 1/5, avg loss: 0.3429
epoch 2/5, avg loss: 0.1436
epoch 3/5, avg loss: 0.0981
epoch 4/5, avg loss: 0.0741
epoch 5/5, avg loss: 0.0587


In [ ]:

# check accuracy on images the model has never seen

model.eval()

correct = 0
total = 0

# no need for gradients here, just checking, not training
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        # highest score wins
        _, predicted = torch.max(outputs, dim=1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"test accuracy: {accuracy:.2f}%")

test accuracy: 97.70%


In [ ]:

# quick look at some predictions

import matplotlib.pyplot as plt

images, labels = next(iter(test_loader))
images_device = images.to(device)

model.eval()
with torch.no_grad():
    outputs = model(images_device)
    _, predicted = torch.max(outputs, dim=1)

fig, axes = plt.subplots(1, 8, figsize=(16, 2))

for i in range(8):
    img = images[i][0]  # drop the channel dim, just 28x28 now
    true_label = labels[i].item()
    predicted_label = predicted[i].item()

    axes[i].imshow(img, cmap="gray")
    axes[i].set_title(f"true: {true_label}\npred: {predicted_label}")
    axes[i].axis("off")

plt.tight_layout()
plt.show()

## Results

Two hidden layers, trained on 60k digit images, ~97% accuracy on the 10k test images.

- neurons/weights = the `nn.Linear` layers
- activation = ReLU
- loss = cross entropy
- gradient descent = Adam, with `loss.backward()` doing all the gradient math for us

Same building blocks (layers, activation, loss, optimizer) show up again in every project after this one.

## Notes

- Learning rate - choose it such that you find optimal, not too high, not too small. Need to skip small bumps but also not skip minimas.
- Randomization of initial weights gives slightly different results each time.